# Modeling (Projet 6)

## Section Imports

In [1]:
import pandas as pd
import numpy as np
import gc
from contextlib import contextmanager
import time
import mlflow
import mlflow.lightgbm
from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, recall_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('mlflow').setLevel(logging.ERROR)


## Config MLFlow

In [2]:

# Config MLFlow
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("home_credit_scoring")

mlflow.sklearn.autolog(log_models=False)
mlflow.xgboost.autolog(log_models=False,)
mlflow.lightgbm.autolog(log_models=False)

## Chargement Data (../input/data_clean.parquet)

In [2]:

# Chargement data
# df = pd.read_csv('../input/data_clean.csv')
df = pd.read_parquet('../input/data_clean.parquet')
train_df = df[df['TARGET'].notnull()]
test_df = df[df['TARGET'].isnull()]
print(f"Train: {train_df.shape}, Test: {test_df.shape}")

Train: (307507, 797), Test: (48744, 797)


In [4]:
df.head(30).to_csv("../input/sample2.csv", index=False)

In [3]:
print([col for col in df.columns if df[col].dtype == 'object'])

[]


## Fonction cv & compare models

In [ ]:
def format_duration(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    parts = []
    if h > 0: parts.append(f"{h} hr{'s' if h > 1 else ''}")
    if m > 0: parts.append(f"{m} mn{'s' if m > 1 else ''}")
    parts.append(f"{s} sec{'s' if s > 1 else ''}")
    return ' '.join(parts)

def business_cost(y_true, y_pred, threshold=0.5):
    y_class = (y_pred >= threshold).astype(int)
    fn = ((y_true == 1) & (y_class == 0)).sum()
    fp = ((y_true == 0) & (y_class == 1)).sum()
    return 10 * fn + 1 * fp

def cv_compare(models, infos="None", threshold=0.5):

    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=1001)
    all_results = {}

    for model_name, model in models.items():
        print(f"\n{'='*20}{model_name} Start{'='*20}")
        t0 = time.time()
        
        with mlflow.start_run(run_name=model_name):
            mlflow.set_tags({
                            "infos": infos,
                            "classification_threshold": threshold   
                            })
            oof_preds = np.zeros(train_df.shape[0])
            auc_scores, recall_scores, f1_scores, cost_scores = [], [], [], []

            for n_fold, (train_idx, valid_idx) in enumerate(folds.split(train_df[feats], train_df['TARGET'])):
                train_x = train_df[feats].iloc[train_idx]
                train_y = train_df['TARGET'].iloc[train_idx]
                valid_x = train_df[feats].iloc[valid_idx]
                valid_y = train_df['TARGET'].iloc[valid_idx]

                if model_name in ("xgboost"):   model.fit(train_x, train_y, eval_set=[(valid_x, valid_y)], verbose=False)
                elif model_name in ("lightgbm"):   model.fit(train_x, train_y, eval_set=[(valid_x, valid_y)])
                else:                                       model.fit(train_x, train_y)

                oof_preds[valid_idx] = model.predict_proba(valid_x)[:, 1]

                fold_auc    = roc_auc_score(valid_y, oof_preds[valid_idx])
                fold_recall = recall_score(valid_y, (oof_preds[valid_idx] >= threshold).astype(int))
                fold_f1     = f1_score(valid_y, (oof_preds[valid_idx] >= threshold).astype(int))
                fold_cost   = business_cost(valid_y, oof_preds[valid_idx])

                auc_scores.append(fold_auc)
                recall_scores.append(fold_recall)
                f1_scores.append(fold_f1)
                cost_scores.append(fold_cost)

                print(f"Fold {n_fold+1} — AUC: {fold_auc:.4f} | Recall: {fold_recall:.4f} | F1: {fold_f1:.4f} | Cost: {fold_cost:.0f}       Time since start: {time.time() - t0:.0f}s")
                mlflow.log_metric(f'auc_fold_{n_fold+1}',    fold_auc)
                mlflow.log_metric(f'recall_fold_{n_fold+1}', fold_recall)
                mlflow.log_metric(f'f1_fold_{n_fold+1}',     fold_f1)
                mlflow.log_metric(f'cost_fold_{n_fold+1}',   fold_cost)

                del train_x, train_y, valid_x, valid_y
                gc.collect()

            # Moyennes
            auc_mean    = np.mean(auc_scores)
            recall_mean = np.mean(recall_scores)
            f1_mean     = np.mean(f1_scores)
            cost_mean   = np.mean(cost_scores)

            print(f"\nMean — AUC: {auc_mean:.4f} | Recall: {recall_mean:.4f} | F1: {f1_mean:.4f} | Cost: {cost_mean:.0f}")
            mlflow.log_metric('auc_mean',    auc_mean)
            mlflow.log_metric('recall_mean', recall_mean)
            mlflow.log_metric('f1_mean',     f1_mean)
            mlflow.log_metric('cost_mean',   cost_mean)

            # AUC overall OOF
            overall_auc = roc_auc_score(train_df['TARGET'], oof_preds)
            mlflow.log_metric('auc_overall', overall_auc)
            print(f"Overall OOF AUC: {overall_auc:.4f}")
            
            print(f"Total time for {model_name}: {format_duration(time.time() - t0)}")

            all_results[model_name] = {'auc': overall_auc, 'cost': cost_mean}

    return all_results

## Main

In [5]:
INFOS = "Test des meilleurs modèles sur la totalité du train (scale_pos_weight=12 pour XGBoost)"

SAMPLE_FRAC = 1.0  # % du train — mettre 1.0 pour le dataset complet

FULLTIME_0_05 = 493  # temps total du run complet avec dataset à 5% (LightGBM + XGBoost + RandomForest + LogisticRegression + MLP) pour estimation

skip_models = ["mlp", "random_forest"]   # Modeles à ignorer pour les runs finaux (trop longsavec dataset complet)


In [6]:
if SAMPLE_FRAC < 1.0:
    train_df = train_df.groupby('TARGET', group_keys=False).apply(
        lambda x: x.sample(frac=SAMPLE_FRAC, random_state=1001)
    )
    print(f"Train réduit: {train_df.shape} | Ratio TARGET: {train_df['TARGET'].mean():.4f}")

models = {
    "logistic_regression": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),          # imputation valeurs manquantes par la médiane (pour les modèles ne gèrant pas les NaN)
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(
            penalty='l2',
            C=1.0,
            solver='lbfgs',
            max_iter=1000,
            class_weight='balanced',                # balanced
            random_state=1001,
            n_jobs=-1
        ))
    ]),

    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=None,
        min_samples_split=2,
        min_samples_leaf=1,
        max_features='sqrt',
        class_weight='balanced',                    # balanced
        random_state=1001,
        n_jobs=-1
    ),

    "xgboost": XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.0,
        reg_lambda=1.0,
        early_stopping_rounds=50,
        random_state=1001,
        n_jobs=-1,
        eval_metric='auc',
        scale_pos_weight=12                         # balanced
    ),

    "lightgbm": LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.02,
        num_leaves=34,
        max_depth=8,
        colsample_bytree=0.9497036,
        subsample=0.8715623,
        reg_alpha=0.041545473,
        reg_lambda=0.0735294,
        min_split_gain=0.0222415,
        min_child_weight=39.3259775,
        class_weight='balanced',                    # balanced
        random_state=1001,
        n_jobs=-1,
        verbose=-1,
        metric='auc'
    ),

    "mlp": Pipeline([
        ('imputer', SimpleImputer(strategy='median')),  # imputation valeurs manquantes par la médiane (pour les modèles ne gèrant pas les NaN)
        ('scaler', StandardScaler()),
        ('clf', MLPClassifier(
            hidden_layer_sizes=(128, 64),
            activation='relu',
            solver='adam',
            alpha=0.0001,
            batch_size='auto',
            learning_rate='constant',
            learning_rate_init=0.001,
            max_iter=100,
            random_state=1001
        ))
    ]),
}

models_to_run = {k: v for k, v in models.items() if k not in skip_models}

print(f"\n{'*'*20}{INFOS}{'*'*20}")
print(f"Temps total estimé: {format_duration((FULLTIME_0_05*SAMPLE_FRAC/0.05)*(len(models_to_run)/len(models)))}")

feats = [f for f in train_df.columns if f not in ['TARGET', 'SK_ID_CURR']]
results = cv_compare(models_to_run, infos=INFOS, threshold=0.5)

# Meilleur modèle selon l'AUC (à maximiser)
best_model_name = max(results, key=lambda k: results[k]['auc'])
print(f"\nBest model vs AUC: {best_model_name} — Cost: {results[best_model_name]['cost']:.0f} | AUC: {results[best_model_name]['auc']:.4f}")

# Meilleur modèle selon coût métier (à minimiser)
best_model_name = min(results, key=lambda k: results[k]['cost'])
print(f"\nBest model vs coût métier: {best_model_name} — Cost: {results[best_model_name]['cost']:.0f} | AUC: {results[best_model_name]['auc']:.4f}")


********************Test des meilleurs modèles sur la totalité du train (scale_pos_weight=12 pour XGBoost)********************
Temps total estimé: 1 hr 38 mns 36 secs

====================logistic_regression Start====================
Fold 1 — AUC: 0.7665 | Recall: 0.6981 | F1: 0.2763 | Cost: 31646       Time since start: 569s
Fold 2 — AUC: 0.7740 | Recall: 0.6918 | F1: 0.2791 | Cost: 31516       Time since start: 1025s
Fold 3 — AUC: 0.7746 | Recall: 0.7084 | F1: 0.2836 | Cost: 30804       Time since start: 1548s
Fold 4 — AUC: 0.7741 | Recall: 0.7053 | F1: 0.2808 | Cost: 31102       Time since start: 1875s
Fold 5 — AUC: 0.7731 | Recall: 0.6959 | F1: 0.2771 | Cost: 31614       Time since start: 2212s

Mean — AUC: 0.7725 | Recall: 0.6999 | F1: 0.2794 | Cost: 31336
Overall OOF AUC: 0.7725
Total time for logistic_regression: 36 mns 52 secs

====================xgboost Start====================
Fold 1 — AUC: 0.7799 | Recall: 0.6441 | F1: 0.3012 | Cost: 30741       Time since start: 239s
Fol